# Project Overview
Throughout this course, you will build an AI-powered retail business with three components working together to create busniess value. The three components are as follows:

1. **🎯 Intelligent Marketing Agent (Project 1 - This Module)**
   - **Goal**: Create the perfect email for each customer and send it. 
   - **Core Idea**: The quality of the email determines how probable clients will visit your shop.
   - **Task**: Implement an LLM-based Intelligent Marketing Agent which learns the optimal messaging strategies from the customer engagement history
   - **Evaluation Metric**: Number of website visits per day.

2. **🛒 AI-Powered Recommendation System (Project 2)**
   - **Goal**: Recommend the most relevant items for each customer visiting the shop.
   - **Core Idea**: The affinity between customers and items should depend on some (unknown) factors and purchase history
   - **Task**: Implement a recommendation algorithm which learns the underlying affinity mechanism from available information
   - **Evaluation Metric**: Sum of order revenue per day.

3. **🚚 Reinforcement Learning Delivery Optimization (Project 3)**
   - **Goal**: Optimizes delivery plans for cost, efficiency, and customer satisfaction
   - **Core Idea**: Minimize the total expenses of the fleet while respecting the preferred delivery time of each customer
   - **Task**: Train a reinforcement learning algorithm which solves the Capacitated Vehicle Routing Problem with Time Windows (CVRP-TW) 
   - **Evaluation Metric**: Total cost of the dispatchment plan.

When powered by these AI components, an e-commerce platform can boost revenue by sending out targeted ads to increase click-through rate (CTR), performing individualized recommendations to increase sales, and optimizing delivery plans to minimize fuel consumption while adhering to customer's perferred delivery time.

In order to facilitate bookkeeping and learning, we will maintain a database consisting of customers, items, as well as past website visits and order placements. The customer behavior and preferences will be simulated by a World Simulator, which we regard as a proxy of the real world. The interactions between all the components are summarized in the figure below.

<center>
<img src="ainit-project-overview.png" alt="ainit-project-overview" width="800"/>
</center>



# Project Part 1 - Intelligent Marketing Agent

In Project 1, you will develop an Intelligent Marketing Agent (IMA) that sends out personalized promotion emails to customers. 

## Customer Segmentation
We will assume that according to some marketing research, it is reasonable to classify each customer into one of the following segments based on average order value (AOV) and purchase history:

<center>

Customer Segment  | Code |  Description          | Marketing Goal  |    
----------------|--|---------------------|------------|
🆕 Newcomers  |  N  |  New users who have not made any purchase yet| Conversion and trust-building by showcasing value and providing a low-risk incentive to encourage the first purchase |
💎 High-Value Customers | H| Top buyers with a high AOV| Focus on retention by making these customers feel recognized for their high spending to reinforce their loyalty and encourage continued large purchases |
🔄 Regular Customers | R | Steady buyers with a moderate AOV |  Consistent engagement and nurturing by keeping them interested with updates on new arrivals and showing appreciation to maintain their shopping habit and build a stronger relationship.|
💰 Low-Value Customers | L |Price-sensitive customers with a low AOV |  Increase their AOV with bundles or a free shipping threshold to encourage them to add more to their cart |
⚠️ At-Risk Customers | A | Inactive users who haven't made purchases in a while | Urgent re-engagement by acknowledging their absence and providing a compelling reason for them to return|

</center>


We further assume that for each customer segment, there is some most effective marketing hook which maximize the probability of the customer visiting the website, based on the characteristic of the segment. The goal of this project is to uncover the hidden optimal marketing hook for each customer segment. Some possible marketing hooks are as follows:

<center>

Customer Segment  | Code |  Example hooks          | 
----------------|--|---------------------|
🆕 Newcomers  |  N  |  Welcome! A special offer awaits inside.|
💎 High-Value Customers | H|  A special gift for our top customers.|
🔄 Regular Customers | R | A thank-you treat for our regulars.|
💰 Low-Value Customers | L | Bundle your favorites and save more.|
⚠️ At-Risk Customers | A | We’ve missed you. Here’s a gift.|

</center>

The customer segments are determined by a fixed set of rules based on the purchase history and can change as time goes on. For simplicity, we keep the customer segment fixed in this project, and the values can be accessed from the customers table in the database.


## Format of Marketing Emails 
To achieve personalized marketing, the IMA sends out a marketing email $e=(h,r)$ consisting of a marketing hook $h$ followed by personalized product recommendations $r$ obtained from the recommendation system. Here, $h$ and $r$ are strings and we concatenate them to form the email $e$. The quality of the email is determined by how similar $h$ is to the optimal marketing hook of the corresponding segment as well as how appealing the recommendations $r$ are to the customer. Since we have not implemented the recommendation system, in this part we assume that the recommendation has no effect on the quality, and we always recommend some random (or fixed) set of products in the email.



## Scoring mechanism of marketing hook
To uncover the (unknown) optimal marking hook $h^*$ for each segment, we seek to find a string $h$ which is as semantically close to $h^*$ as possible. An effective way to represent a string in a semantically meaningful way is to convert it into a (high-dimensional) vector called  *embedding*. These embeddings are usually obtained from some neural networks (typically *transformers*) trained on a large dataset. In this project we will use an encoder model called  $\text{BERT}$: given a string $s$, its $\text{BERT}$ embedding would be a vector which we denote as $\text{BERT}(s)$.

Once we translate strings into vectors, we can use similarity measures between two vectors as a proxy for string semanic similarity. A standard choice (which we adopt) is the so-called *cosine similarity*, i.e. the inner product of the normalized vectors:
$$
\texttt{cos-sim}(v,w):= \begin{cases} \frac{\langle v,w\rangle}{\|v\| \|w\|} & \text{if } v\neq 0\neq w\\
0 & \text{otherwise}.
\end{cases}
$$
Here, $v$ and $w$ are two vectors of the same dimension, $\langle \cdot,\cdot\rangle$ denotes the inner product, and $\|\cdot\|$ denotes the Euclidean $2$-norm. As a dot product, this value lies between $-1$ and $1$.

Thus, for a customer whose optimal hook is $h^*$, the score representing the quality of a hook $h$ is given by $\texttt{cos-sim}(\text{BERT}(h),\text{BERT}(h^*))$. This score in term determines the probability of the customer visiting our website: the higher it is, the more likely the customer will visit.

## How to find the optimal hooks? Introducing the OPRO framework
We have translated the task of finding a particular string (optimal marketing hook) into an optimization problem. Formally, for each customer segment we want to solve for the string $\hat{h}$ maximizing the cosine similarity, namely
$$
\hat{h}:=\argmax_{h} \texttt{cos-sim}(\text{BERT}(h),\text{BERT}(h^*)).
$$

How can this be done? One idea is to leverage the capabilities of LLMs and use them as optimizer, proposed in the paper [Large Language Models as Optimizers](https://arxiv.org/abs/2309.03409). This paper introduce the  **OPRO (Optimization by PROmpting)** framework, in which an LLM solves an optimization problem by iteratively improving upon itself based on performance feedback.


The interaction between our marketing agent and the customers runs in cycles and is managed by the World Simulator (WS): 
1. Every day, the Intelligent Marketing Agent creates and sends out an email for each customer in the registry. 
2. Based on the email hook received, the WS simulates whether or not each customer decides to visit the shop on that day.
3. At the end of the day we count the number of customers of each segment that came and save the results to the database. 
4. After 5 days, we take the average visit rates for each segment, which will be the score of the currently used email hook.
5. We then run one OPRO step to make the LLM generate an even better hook.

For example, assume we have $8$ customers in a certain customer segment, and at the start of this cycle we already have several hooks $h_1,\ldots, h_k$ with scores $s_1,\ldots, s_k$. Following the OPRO logic, we would prompt the LLM to come up with a new and hopefully better-performing hook by feeding the existing hooks and their scores. Here is an example of how we can prompt:

<center>
<img src="opro-dummy-example.png" alt="opro-dummy-example" width="800"/>
</center>

Once we get the new hook $h$ as output, we use $h$ in the marketing emails for all $8$ customers in the segments for the following $5$ days and will get $8*5=40$ responses from the WS. If we observe $24$ visits in total, the score of $h$ would be $24/40=0.6$. We then keep the best performing $k$ hooks out of $h,h_1,\ldots, h_k$ and repeat the process.


Now we are ready to get started!

Tips when working in Jupyter notebooks: 
1. When the cell output is too long, you can collapse it or make it scrollable by double-clicking the left side of the cell.
2. You can collapse an entire cell by clicing on the color bar on the very left.

In [1]:
# Install dependencies
%pip install -r requirements.txt

ERROR: Could not find a version that satisfies the requirement torch==2.6.0 (from versions: 2.9.0)

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for torch==2.6.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import required modules
from app.world_sim_p1 import WorldSimulatorP1, LoggingConfig, MarketingSession
from services.llm_service import LLMService
from system_config.system_parameters import SystemParameters
from agents.hooks import HookEvaluator
import numpy as np
import pandas as pd
import ast
import os
import sys
from typing import Tuple

print("✅ All modules imported successfully")

✅ All modules imported successfully


# 1. Database Setup

In this part we will set up the data to work with, mainly the customer segments and associated marketing hooks. We have access to a customer registry in our system from which we can simply read which segment they belong to, but the most effective marketing hooks per segment is unknown to us and we hope to discover them via OPRO.

You do not need to worry about other details and no implementation from your side is required in this part.
Click on the very left side of the cell to hide it.

In [3]:
def load_data_from_files() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load data from TSV files in the data folder."""
    data_dir = os.path.join(os.getcwd(), "data")
    
    try:
        # Check if data files exist
        customers_file = os.path.join(data_dir, "customers.tsv")
        products_file = os.path.join(data_dir, "products.tsv")
        hooks_file = os.path.join(data_dir, "optimal_hooks.tsv")
        
        if not os.path.exists(customers_file):
            raise FileNotFoundError(f"customers.tsv not found at {customers_file}")
        if not os.path.exists(products_file):
            raise FileNotFoundError(f"products.tsv not found at {products_file}")
        if not os.path.exists(hooks_file):
            raise FileNotFoundError(f"optimal_hooks.tsv not found at {hooks_file}")
        
        # Load customers data
        customers_df = pd.read_csv(customers_file, sep='\t')
        
        # Load products data  
        products_df = pd.read_csv(products_file, sep='\t')
        
        # Load optimal hooks data
        hooks_df = pd.read_csv(hooks_file, sep='\t')
        
        # Basic validation
        if customers_df.empty:
            raise ValueError("customers.tsv is empty")
        if products_df.empty:
            raise ValueError("products.tsv is empty")
        if hooks_df.empty:
            raise ValueError("optimal_hooks.tsv is empty")
            
        return customers_df, products_df, hooks_df
        
    except Exception as e:
        print(f"Error loading data files: {e}")
        raise

def create_system_parameters_from_data(customers_df: pd.DataFrame, products_df: pd.DataFrame, hooks_df: pd.DataFrame):
    """Create system parameters from loaded data."""
    from system_config.system_parameters import SystemParameters
    import numpy as np
    
    C = len(customers_df)  # Number of customers
    I = len(products_df)   # Number of items
    K = 5   # Items displayed on website
    D = 5   # 5-dimensional feature space (as specified in readme)
    
    return SystemParameters(K=K)

print("✅ Data loading functions defined")

✅ Data loading functions defined


In [4]:
# Load real data from TSV files - MATCHES main_p1.py
print("📂 Loading data from TSV files...")
customers_df, products_df, hooks_df = load_data_from_files()

# Limit to first 40 customers for faster execution - MATCHES main_p1.py
customers_df = customers_df.head(40)
print(f"✅ Loaded {len(customers_df)} customers and {len(products_df)} products (limited to 40 customers)")

# Set random seed for reproducible results - MATCHES main_p1.py
import numpy as np
np.random.seed(42)
print("🎲 Random seed set to 42 for reproducible results")

📂 Loading data from TSV files...
✅ Loaded 40 customers and 50 products (limited to 40 customers)
🎲 Random seed set to 42 for reproducible results


In [5]:
# Setup real data in database - MATCHES main_p1.py exactly

from peewee import IntegrityError


def setup_real_data(customers_df: pd.DataFrame, products_df: pd.DataFrame) -> None:
    """Setup real customers and products from TSV data using Peewee models."""
    from models.models import Customer, CustomerSegment
    from models.models import Item

    # Insert customers from TSV data
    for _, row in customers_df.iterrows():
        # Gender is already in string format
        gender = row['Gender']
        
        # Generate random location coordinates (could be enhanced with real locations)
        location = (np.random.uniform(-100, 100), np.random.uniform(-100, 100))
        
        # Generate default satisfaction score based on income and segment
        # Higher income customers tend to have higher base satisfaction
        income_factor = min(row['Income'] / 300000, 1.0)  # Normalize income to 0-1 scale
        segment = row['Segment']

        # Assign satisfaction based on customer segment
        segment_satisfaction = {
            'H': 0.8,  # High-value customers
            'R': 0.7,  # Regular customers
            'N': 0.6,  # Newcomers
            'L': 0.5,  # Low-value customers
            'A': 0.4   # At-risk customers
        }
        base_satisfaction = segment_satisfaction.get(segment, 0.6)
        satisfaction = min(base_satisfaction + (income_factor * 0.2), 1.0)
        
        # Parse feature vector from the TSV data
        feature_vector = ast.literal_eval(row['FeatureVector'])
        feature_vector_array = np.array(feature_vector)

        # Create customer using Peewee model
        Customer.create(
            cid=row['CID'],
            name=row['Name'], 
            age=row['Age'],
            gender=row['Gender'],
            location_x=location[0],
            location_y=location[1],
            job_type=row['JobType'],
            income=row['Income'],
            segment=row['Segment'],
            family_status=row['FamilyStatus'],
            feature_vector=feature_vector_array.tolist(),
            satisfaction=satisfaction,
            visit_probability=0.1,  # Default value
            session_probability=0.3,  # Default value
            trigger_keywords=[]  # Default empty list
        )
    
    # Insert products from TSV data
    for _, row in products_df.iterrows():
        # Use cost from TSV if available, otherwise calculate as 50% of price
        cost = row['Cost'] if 'Cost' in row and pd.notna(row['Cost']) else row['Price'] * 0.5
        
        # Generate random stock level between 10-100
        stock_level = np.random.randint(10, 101)
        
        # Default discounted to False since it's not in TSV
        discounted = False

        # Parse feature vector from the TSV data
        feature_vector = ast.literal_eval(row['FeatureVector'])
        feature_vector_array = np.array(feature_vector)

        # Create item using Peewee model
        Item.create(
            pid=row['PID'],
            name=row['Name'],
            description=row['Description'],
            price=row['Price'],
            discounted=discounted,
            cost=cost,
            stock_level=stock_level,
            feature_vector=feature_vector_array.tolist()
        )

def setup_real_transactions(customers_df: pd.DataFrame, products_df: pd.DataFrame) -> None:
    """Setup real transactions from TSV data using Peewee models."""
    from models.models import Transaction
    from datetime import datetime
    
    try:
        # Load transactions data
        data_dir = os.path.join(os.getcwd(), "data")
        transactions_file = os.path.join(data_dir, "transactions.tsv")
        
        if not os.path.exists(transactions_file):
            print("⚠️  Warning: transactions.tsv not found, skipping transaction loading")
            return
            
        transactions_df = pd.read_csv(transactions_file, sep='\t')
        print(f"📊 Loading {len(transactions_df)} transactions...")
        
        # Create price lookup from products
        price_lookup = {row['PID']: row['Price'] for _, row in products_df.iterrows()}
        
        # Insert transactions
        for _, row in transactions_df.iterrows():
            # Calculate revenue from price lookup
            revenue = price_lookup.get(row['PID'], 0.0) * row['Quantity']
            
            Transaction.create(
                customer_id=row['CID'],
                item_id=row['PID'],
                transaction_date=datetime.now(),
                quantity=row['Quantity'],
                revenue=revenue
            )
            
    except Exception as e:
        print(f"⚠️  Error loading transactions: {e}")

# Reset database to ensure clean state - MATCHES main_p1.py
print("🧹 Resetting database...")
from models.database import reset_database
reset_database()

# Initialize Peewee database and create tables - MATCHES main_p1.py
print("🔧 Initializing database...")
from models.database import initialize_database
initialize_database()
print("✅ Initialized Peewee database with updated schema")

# Execute the setup functions
print("🔧 Setting up real data in database...")
try:
    setup_real_data(customers_df, products_df)
except IntegrityError as e:
    print(f"⚠️ Error during setup_real_data: {e}")
    print(f"⚠️ Data was likely already inserted, continuing...")
print("🔧 Setting up real transactions from TSV data...")
setup_real_transactions(customers_df, products_df)

print("✅ Real data setup complete")

🧹 Resetting database...
🔧 Initializing database...
✅ Initialized Peewee database with updated schema
🔧 Setting up real data in database...
🔧 Setting up real transactions from TSV data...
📊 Loading 639 transactions...
⚠️  Error loading transactions: 'PID'
✅ Real data setup complete


## 2. Configure LLM Service

In this part we will set up the LLM we will use as the base of our marketing agent. It will be responsible for creating and optimizing marketing hooks.

You do not need to worry about the details and no implemention is required from your side.

In [6]:
# Download model
from huggingface_hub import hf_hub_download
model_path = hf_hub_download(repo_id="Qwen/Qwen2.5-1.5B-Instruct-GGUF",
                                                          filename="qwen2.5-1.5b-instruct-q5_k_m.gguf",
                                                         )
model_path

'/Users/santiago/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct-GGUF/snapshots/91cad51170dc346986eccefdc2dd33a9da36ead9/qwen2.5-1.5b-instruct-q5_k_m.gguf'

In [7]:
# Initialize LLM service with Qwen model - MATCHES main_p1.py
print("🤖 Initializing LLM service with Qwen 2.5 model...")

if not os.path.exists(model_path):
    raise FileNotFoundError(f"❌ Qwen model file not found at {model_path}")

from services.llm_service import LlamaFileProvider
llm_provider = LlamaFileProvider(model_path=model_path, n_ctx=8192, verbose=False)  # MATCHES main_p1.py
llm_service = LLMService(provider=llm_provider)

# Test LLM service
print("🧪 Testing LLM service...")
test_response = llm_service.generate_text("Write a short marketing hook for premium coffee.")
print(f"✅ LLM test response: {test_response}")

🤖 Initializing LLM service with Qwen 2.5 model...


llama_context: n_ctx_per_seq (8192) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
ggml_metal_init: skipping kernel_get_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_set_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_c4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_1row              (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_l4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_bf16                  (not supported)
ggml_metal_init: skipping kernel_mul_mv_id_bf16_f32                (not supported)
ggml_metal_init: skipping kernel_mul_mm_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mm_id_bf16_f16                (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h64 

🧪 Testing LLM service...
✅ LLM test response: Discover the ultimate taste experience with our premium coffee. Unleash your inner coffee connoisseur with our meticulously crafted blends that tantalize your senses. From rich and bold to smooth and creamy, we have it all. Embrace the warmth of our beans in every sip, and indulge in the luxury of a premium coffee experience. Order now and elevate your coffee game.


## 3. Initialize World Simulator Components

This part sets up the system parameters and logging configurations.
You do not need to worry about the details and no implemention is required from your side.

In [8]:
# Create system parameters from real data
print("⚙️ Creating system parameters from real data...")
system_parameters = create_system_parameters_from_data(customers_df, products_df, hooks_df)

# Create optimal hooks dictionary from data
optimal_hooks = {}
for _, row in hooks_df.iterrows():
    optimal_hooks[row['Segment']] = row['Hook']

print(f"✅ Loaded optimal hooks for {len(optimal_hooks)} segments:")
for segment, hook in optimal_hooks.items():
    print(f"   • {segment}: {hook[:60]}...")

# Initialize HookEvaluator
print("🎣 Initializing Hook Evaluator...")
hook_evaluator = HookEvaluator(optimal_hooks=optimal_hooks)
print("✅ Hook Evaluator initialized with BERT embeddings")

# Configure logging - customize what output you want to see
logging_config = LoggingConfig(
    daily_progress=False,          # Show "Running simulation day X/Y"
    marketing_hooks=False,         # Show generated hooks for segments
    marketing_individual=False,    # Show individual customer marketing results
    shopping_phase=False,          # Show "Shopping Phase - Day X"
    accuracy_daily=False,          # Show daily accuracy calculations
    accuracy_cycle=False,          # Show OPRO cycle progress
    opro_optimization=False,       # Show OPRO optimization process
    opro_prompts=False,           # Show new optimized prompts
    opro_fallbacks=False,         # Show OPRO fallback messages
    day_advancement=False,        # Show "Advanced to Day X"
    final_history=False,          # Show OPRO history at end
    setup_messages=False          # Show initialization messages
)

print("✅ Logging configuration set")

⚙️ Creating system parameters from real data...
✅ Loaded optimal hooks for 5 segments:
   • N: Welcome aboard! Discover the quality our customers love and ...
   • H: Your orders deserve a reward! As a top customer, receive a c...
   • R: Just for our regulars! See what's new and trending this week...
   • L: Great finds, even better value! Explore our affordable best-...
   • A: It's been a while, and we've missed you! Come see what's new...
🎣 Initializing Hook Evaluator...
✅ Hook Evaluator initialized with BERT embeddings
✅ Logging configuration set


## 4. Implement key parts of the World Simulator
### 4.1 Complete the HookEvaluator class

The `HookEvaluator` class handles the task of evaluating a marketing hook relative to the optimal hook for each customer segment.

**Your Task**: Implement the core methods of the HookEvaluator class:
- `evaluate_hook_quality_with_embedding`: Calculate similarity between two vectors
- `evaluate_hook_quality`: Evaluate hook quality using pre-computed embeddings

Replace `raise NotImplementedError` with your implementation. The simulation will not work until you implement these functions.

**Hints**: You may find the following useful:
- `np`: NumPy library for vector operations
- `self._optimal_embeddings`: Dict mapping segments to optimal hook embeddings
- `self.encoder.encode(str)`: encode a string as vector

In [9]:
# EXERCISE: Implement your custom hook evaluator methods

def custom_cosine_similarity(self, vec1, vec2):
    """
    Custom cosine similarity implementation.
    
    Args:
        vec1: First vector (numpy array)
        vec2: Second vector (numpy array)
        
    Returns:
        Similarity score between 0.0 and 1.0
    
    TODO: Implement your custom similarity calculation here.
    
    Ideas to try:
    - Standard cosine similarity: dot(normalized_vec1, normalized_vec2)
    - Euclidean distance converted to similarity: 1.0 / (1.0 + distance)
    - Manhattan distance: 1.0 / (1.0 + manhattan_distance)
    - Custom weighted similarity based on specific dimensions
    
    Hint: Use np.linalg.norm() for vector normalization
    Hint: Use np.dot() for dot product
    """
    # YOUR IMPLEMENTATION HERE
    raise NotImplementedError("Implement HookEvaluator.custom_cosine_similarity")

def custom_evaluate_hook_quality_with_embedding(self, hook_embedding, customer_segment):
    """
    Custom hook quality evaluation using pre-computed embedding.
    
    Args:
        hook_embedding: Pre-computed normalized embedding of the hook (numpy array)
        customer_segment: Customer segment (N, H, A, L, R, or Default)
        
    Returns:
        Hook quality score between 0.0 and 1.0
    
    TODO: Implement your custom hook quality evaluation here.
    
    Available data:
    - self._optimal_embeddings: Dict mapping segments to optimal hook embeddings
    - self.cosine_similarity(): Your custom similarity function
    
    Ideas to try:
    - Compare hook_embedding with optimal embedding for the segment
    - Apply segment-specific weighting factors
    - Combine multiple similarity metrics
    - Add bonus/penalty based on segment characteristics
    
    Hint: Get optimal embedding with self._optimal_embeddings.get(customer_segment, self._optimal_embeddings["Default"])
    Hint: Use self.cosine_similarity(hook_embedding, optimal_embedding)
    """
    # YOUR IMPLEMENTATION HERE
    raise NotImplementedError("Implement HookEvaluator.custom_evaluate_hook_quality_with_embedding")

print("\n⚠️  WARNING: Functions are currently empty")
print("   You need to implement them before running the simulation!")

# Apply monkey patching to replace the original methods
print("🔧 Applying custom hook evaluator methods...")
HookEvaluator.cosine_similarity = custom_cosine_similarity
HookEvaluator.evaluate_hook_quality_with_embedding = custom_evaluate_hook_quality_with_embedding

print("✅ Custom methods applied to HookEvaluator class")
print("   → cosine_similarity: Custom implementation active")
print("   → evaluate_hook_quality_with_embedding: Custom implementation active")
print("\n💡 The hook_evaluator instance will now use your custom methods!")


⚠️  WARNING: Functions are currently empty
   You need to implement them before running the simulation!
🔧 Applying custom hook evaluator methods...
✅ Custom methods applied to HookEvaluator class
   → cosine_similarity: Custom implementation active
   → evaluate_hook_quality_with_embedding: Custom implementation active

💡 The hook_evaluator instance will now use your custom methods!


### 4.2 Complete the MarketingSession class


The `MarketingSession` class handles the following functionalities:
- Customer segmentation using transaction history (assumed to be static for this project)
- Marketing hook quality evaluation using cosine similarity with optimal hooks
- Customer visit probability updates based on marketing effectiveness
- Session history persistence for performance tracking

The universe uses the following from the `MarketingSession` class to orchestrate a full marketing interaction:
- `run_session`: This method runs the daily workflow of the marketing agent on one customer:
  1. Compute/Get the segment of the customer
  2. Get the current hook for the segment
  3. Run one marketing step `run_step`
  4. Update the probability of this customer visiting the website based on hook quality
  5. Store the interaction into `self.session_history` as a dict
  6. Return `self.session_history`
- `run_step`: This method runs the core marketing action on one customer:
  0. Input: session history, hook to use, segment of this customer, instruction for LLM
  1. Craft a marketing email based on the input using `self.marketing_agent.createMessage`
  2. Observe customer response using `self.customer_response.create_response`
  3. Evaluate the quality of the hook using `self.evaluate_hook_quality`

**Your Task**
- Use the custom hook evaluator you implemented above in the `run_step` method to assign a `hook_quality_score` to the hook in the step.

In [10]:
def run_session(self):
    """Run the complete marketing session."""
    # 1. Classification of customer segment
    customer_segment = self.classify_customer_segment() # CustomerSegment.HIGH_VALUE_CUSTOMERS.value

    # 3. Use pre-generated hook from WorldSimulator
    selected_hook = self.current_hooks.get(customer_segment)

    if not selected_hook:
        # Fallback to optimal hooks if current_hooks is empty
        selected_hook = self.optimal_hooks.get(customer_segment, self.optimal_hooks.get("Default", "Special offers just for you!"))

    marketing_email, client_response, hook_quality_score = self.run_step(self.session_history, selected_hook, customer_segment, self.llm_instruction_prompt.get(customer_segment))

    # Update visit probability
    self.update_customer_visit_proba(hook_quality_score)

    self.session_history.append({
        'marketing_email': marketing_email,
        'client_response': client_response,
        'customer_segment': customer_segment,
        'email_hook': selected_hook,
        'llm_hook_instruction': self.llm_instruction_prompt.get(customer_segment)
    })

    return self.session_history

def run_step(self, session_history, email_hook=None, customer_segment=None, llm_hook_instruction=None):
    """Run a single step of the marketing session."""
    # Marketing agent creates MarketingEmail for client
    # Pass the email hook, segment, and LLM instruction to potentially customize the message
    marketing_email = self.marketing_agent.createMessage(self.client, email_hook, customer_segment, llm_hook_instruction)

    # Client creates response based on the marketing email object
    client_response = self.customer_response.create_response(marketing_email)

    # Use the hook evaluator to get the score of the hook
    hook_quality_score = 0.5  # Default score
    if hasattr(marketing_email, 'marketing_hook') and marketing_email.marketing_hook:
        # TODO: Use your custom hook evaluator here
        # hook_quality_score = my_custom_hook_evaluator(marketing_email.marketing_hook, customer_segment)
        raise NotImplementedError("Get hook quality score using your custom evaluator")

    return marketing_email, client_response, hook_quality_score

print("\n⚠️  WARNING: Implement the functions before running the simulation!")
# Apply monkey patching to replace the original methods
MarketingSession.run_session = run_session
MarketingSession.run_step = run_step


⚠️  WARNING: Implement the functions before running the simulation!


### 4.3 Initialize WorldSimulator
Now that we have filled in all the missing part of the World Simulator, we will instantiate an `WorldSimulatorP1` object (P1 for project 1) for future use. Simply run the cell below; no implementation is required from your side.

In [11]:
# Initialize World Simulator P1 with simulation parameters - MATCHES main_p1.py
print("🌍 Initializing World Simulator...")

# OPRO Configuration - MATCHES main_p1.py:
# - use_opro2=False: Original OPRO (optimizes LLM instruction prompts)  
# - use_opro2=True:  OPRO2 (optimizes marketing hooks directly)
use_opro2_mode = True  # MATCHES main_p1.py line 360

w = WorldSimulatorP1(
    llm_service=llm_service, 
    system_parameters=system_parameters, 
    hook_evaluator=hook_evaluator, 
    simulation_duration=50,      # 15 days for faster notebook execution (vs 100 in main_p1.py)
    opro_cycle_days=5,          # 5-day OPRO cycles - MATCHES main_p1.py line 363
    logging_config=logging_config,
    use_opro2=use_opro2_mode    # OPRO2 mode - MATCHES main_p1.py line 364
)

opro_mode_name = "OPRO2 (Direct Hook Optimization)" if use_opro2_mode else "Original OPRO (Prompt Optimization)"
print(f"✅ Using optimization mode: {opro_mode_name}")
print(f"✅ World Simulator P1 initialized for {w.simulation_duration} days with {w.opro_cycle_days}-day OPRO cycles")

🌍 Initializing World Simulator...
Cleared Peewee marketing operations
✅ Using optimization mode: OPRO2 (Direct Hook Optimization)
✅ World Simulator P1 initialized for 50 days with 5-day OPRO cycles


### 4.4 OPRO Cycle Accuracy Calculation
In this part, you will implement the methods calculating how performant a marketing hook is over an OPRO 5-day cycle on a per-segment basis. This method directly provides your LLM the performance feedback in order to conduct optimization; in other words, we will feed hook-accuracy pairs into the OPRO framework in order to come up with better hooks.

**Your Task**
- Implement the `calculate_segment_hook_accuracy` method which calculates the accuracy of each hook as the customer visit rate

In [12]:
from typing import Optional, Dict


# EXERCISE: Implement your custom OPRO cycle accuracy methods

def calculate_segment_hook_accuracy(self, day_number=None):
    """
    Calculate hook accuracy for each customer segment based on shop visit rates.
    
    Accuracy is defined as: (number of customers in segment who visited) / (total customers in segment)
    
    Args:
        day_number: Simulation day (defaults to current day)
        
    Returns:
        Dict[str, float]: Dictionary mapping segment to accuracy score (0.0-1.0)
    
    TODO: Implement your custom segment hook accuracy calculation here.
    
    Available data:
    - self.customer_registry: Access to all customer data with segments
    - self.shop_component: Access to shop visit data for any day
    - self.current_simulation_day: Current simulation day if day_number is None
    
    Algorithm:
    1. Get all customers and group them by segment
    2. Count total customers per segment
    3. Get shop visits for the specified day
    4. Count how many customers from each segment visited
    5. Calculate accuracy = visited_count / total_count per segment
    
    Hint: Use self.customer_registry.get_all_customers()
    Hint: Use self.shop_component.get_visits_for_day(day_number)
    Hint: Create a dictionary to track {'total': count, 'visited': count} per segment
    Hint: Return a dict mapping segment -> accuracy (0.0-1.0)
    """
    # YOUR IMPLEMENTATION HERE
    raise NotImplementedError("WorldSimulatorP1.calculate_segment_hook_accuracy")

print("\n⚠️  WARNING: Implement the function before running the simulation!")
# Apply monkey patching to replace the original methods
print("🔧 Applying custom OPRO cycle accuracy methods...")
WorldSimulatorP1.calculate_segment_hook_accuracy = calculate_segment_hook_accuracy

print("✅ Custom methods applied to WorldSimulatorP1 class")
print("   → calculate_segment_hook_accuracy: Custom implementation active")
print("   → calculate_segment_hook_accuracy_differences: Custom implementation active")
print("\n💡 The world simulator instance will now use your custom cycle accuracy methods!")

print("\n📊 OPRO Cycle Accuracy Functions:")
print("   • These functions are crucial for OPRO optimization to work properly")
print("   • They calculate performance improvements between optimization cycles")
print("   • Results drive the prompt/hook optimization process")


⚠️  WARNING: Implement the function before running the simulation!
🔧 Applying custom OPRO cycle accuracy methods...
✅ Custom methods applied to WorldSimulatorP1 class
   → calculate_segment_hook_accuracy: Custom implementation active
   → calculate_segment_hook_accuracy_differences: Custom implementation active

💡 The world simulator instance will now use your custom cycle accuracy methods!

📊 OPRO Cycle Accuracy Functions:
   • These functions are crucial for OPRO optimization to work properly
   • They calculate performance improvements between optimization cycles
   • Results drive the prompt/hook optimization process


### 4.5 Create an OPRO prompt template
In this part, you will implement the OPRO framework -- optimization by prompting -- to optimize for the best marketing hook (per segment). We will solve this problem via prompting our LLM with instructions and hooks we have tried in the past along with their accuracy scores.

**Your Task**
- Implement the `create_meta_prompt` method which provides the prompt to carry out OPRO. This would involve writing a prompt template as a templated string in order to dynamically incorporate past hook-accuracy pairs. 
  
**Hint**
- Read and understand the `create_meta_prompt_example` method as a starting point.


In [13]:
# EXERCISE: Implement 

from dataclasses import dataclass
from models.models import CustomerSegment
from agents.opro import OptimizerEntry2, Opro2
from typing import List

# Feel free to redefine segment_descriptions here if needed
segment_descriptions = {
    CustomerSegment.NEWCOMERS.value: "new customers who have never purchased before",
    CustomerSegment.HIGH_VALUE_CUSTOMERS.value: "customers who make frequent large purchases",
    CustomerSegment.REGULAR_CUSTOMERS.value: "regular customers who occasionally make purchases",
    CustomerSegment.AT_RISK_CUSTOMERS.value: "regular customers who haven't purchased recently or show low satisfaction",
    CustomerSegment.LOW_VALUE_CUSTOMERS.value: "customers who make purchases with a low average value"
}

@dataclass
class OptimizerEntry2:
    """
    Data structure for OPRO optimization entries containing performance metrics
    and associated hooks for a specific customer segment.

    accuracy_score represents the actual visit rate accuracy for the cycle
    (range: 0.0 to 1.0)
    """
    accuracy_score: float    # Visit rate accuracy score (0.0 to 1.0): proportion of customers who visited
    hook: str               # Marketing hook text that achieved this accuracy

def create_meta_prompt_example(self, customer_segment: str, top_performers: List[OptimizerEntry2]) -> str:
    segment_desc = segment_descriptions.get(customer_segment, "general customers")
    # Sort best-first
    top_performers = sorted(top_performers, key=lambda e: e.accuracy_score, reverse=True)

    # Build examples block
    examples_text = []
    for i, entry in enumerate(top_performers, 1):
        examples_text.append(
            f"Example {i} — Visit Rate: {entry.accuracy_score:.3f}\n"
            f'Hook: "{entry.hook}"'
        )
    examples_block = "\n\n".join(examples_text)

    best = top_performers[0]

    # Create meta-prompt template + fill in details
    meta_prompt = f"""
You are a concise marketing copywriter.

TARGET AUDIENCE: {segment_desc}

GOAL: Make a tiny improvement to the BEST hook to increase visit rate.

BEST HOOK (baseline):
- Visit Rate: {best.accuracy_score:.3f}
- Text: "{best.hook}"

OTHER TOP HOOKS (for reference):
{examples_block}

RULES:
- Start from the BEST hook.
- Do NOT rewrite completely. 
- Do NOT explain.

OUTPUT:
Write ONLY the improved hook (one line, no quotes, no extra text)."""

    return meta_prompt

def create_meta_prompt(self, customer_segment: str, top_performers: List[OptimizerEntry2]) -> str:
    # TODO: Implement your custom meta-prompt creation here.
    # You can use create_meta_prompt_example() as a starting point or inspiration.
    # YOUR IMPLEMENTATION HERE
    raise NotImplementedError("Implement Opro2.create_meta_prompt")

# Use the custom create_meta_prompt method
from agents.opro import Opro2
Opro2.create_meta_prompt = create_meta_prompt

try:
    print("📝 Example Meta Prompt Result: This is what the meta-prompt looks like when given a customer segment target + hooks used so far\n")
    print("=" * 25 + " BEGIN META-PROMPT " + "=" * 25)
    example_segment = CustomerSegment.HIGH_VALUE_CUSTOMERS.value
    example_top_performers = [
        OptimizerEntry2(accuracy_score=0.75, hook="Exclusive deals for ourhigh-value customers!"),
        OptimizerEntry2(accuracy_score=0.72, hook="Special offers just for you, our valued customer!"),
        OptimizerEntry2(accuracy_score=0.70, hook="Unlock premium savings today with our exclusive deals!"),
        OptimizerEntry2(accuracy_score=0.68, hook="Don't miss out on special offers tailored for you!"),
        OptimizerEntry2(accuracy_score=0.65, hook="Join our loyalty program for exclusive discounts!")
    ]
    # TODO: Use your own create_meta_prompt implementation here to see how the meta-prompt will be rendered as
    meta_prompt_example = create_meta_prompt_example(None, example_segment, example_top_performers)
    print(meta_prompt_example)
    print("\n=" * 25 + " END META-PROMPT " + "=" * 25)
except Exception as e:
    print(f"⚠️  Error generating example meta-prompt: {e}") 

print("✅ Custom methods applied to Opro class")
print("   → create_meta_prompt: Custom implementation active")
print("\n💡 The OPRO instance will now use your custom optimization methods!")
print("\n⚠️  WARNING: Functions are currently empty (return None)")
print("   You need to implement them before running the simulation!")
print("\n🧠 OPRO Optimization Functions:")
print(" This function implements the meta-prompt creation for OPRO")

📝 Example Meta Prompt Result: This is what the meta-prompt looks like when given a customer segment target + hooks used so far

========================= BEGIN META-PROMPT =========================

You are a concise marketing copywriter.

TARGET AUDIENCE: customers who make frequent large purchases

GOAL: Make a tiny improvement to the BEST hook to increase visit rate.

BEST HOOK (baseline):
- Visit Rate: 0.750
- Text: "Exclusive deals for ourhigh-value customers!"

OTHER TOP HOOKS (for reference):
Example 1 — Visit Rate: 0.750
Hook: "Exclusive deals for ourhigh-value customers!"

Example 2 — Visit Rate: 0.720
Hook: "Special offers just for you, our valued customer!"

Example 3 — Visit Rate: 0.700
Hook: "Unlock premium savings today with our exclusive deals!"

Example 4 — Visit Rate: 0.680
Hook: "Don't miss out on special offers tailored for you!"

Example 5 — Visit Rate: 0.650
Hook: "Join our loyalty program for exclusive discounts!"

RULES:
- Start from the BEST hook.
- Do NOT rewri

## 5. Running OPRO

### 5.1 Display customer data
Before running the optimization, let's have a look at who our customers are. Note that before running the simulation, all visit probabilities are initialized to the default value `0.1`.

In [14]:
# Read all customers and display as DataFrame
print("\n👥 === CUSTOMER DATA ===")
all_customers = w.customer_registry.get_all_customers()

# Create DataFrame from customer data
customer_data = []
for customer in all_customers:
    customer_data.append({
        'CID': customer.cid,
        'Name': customer.name,
        'Age': customer.age,
        'Gender': customer.gender,
        'Job': customer.job_type,
        'Income': f"${customer.income:,}",
        'Segment': customer.segment,
        'Satisfaction': round(customer.satisfaction, 3),
        'Visit_Prob': round(customer.visit_probability, 3)
    })

customers_df_display = pd.DataFrame(customer_data)
print(customers_df_display.to_string(index=False))
print(f"\n✅ Total customers loaded: {len(all_customers)}")
print("=" * 50)


👥 === CUSTOMER DATA ===
 CID            Name  Age Gender                    Job   Income Segment  Satisfaction  Visit_Prob
   0       Liam Chen   45   Male              Architect $250,000       H         0.967         0.1
   1   Olivia Dubois   38 Female            Art Curator $180,000       N         0.720         0.1
   2   Marcus Thorne   52   Male                    CEO $400,000       R         0.900         0.1
   3  Isabella Rossi   33 Female       Fashion Designer $220,000       R         0.847         0.1
   4     Aris Thorne   48   Male                Surgeon $350,000       H         1.000         0.1
   5      Ben Carter   21   Male                Student  $15,000       L         0.510         0.1
   6     Chloe Davis   29 Female       Freelance Writer  $45,000       L         0.530         0.1
   7    Sarah Miller   35 Female                Teacher  $80,000       L         0.553         0.1
   8 David Rodriguez   42   Male         Factory Worker  $55,000       N         0.6

### 5.2 Run simulation & visualize the results

We are finally ready to carry out our marketing campaign and see how it works in action. 
Our marketing agent will perform its workflow while carrying out OPRO behind the scenes to improve its own performance over the course of the simulation.

You don't need to worry about the details in the cell below and can simply run and collapse it to save space.

In [15]:
import math
import time
%matplotlib widget
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as W
from IPython.display import display, clear_output
from collections import deque

# ======================
# 1) UI scaffolding
# ======================
out_plot  = W.Output(layout=W.Layout(border='1px solid #ddd', height='360px'))
out_tbl_1 = W.Output(layout=W.Layout(border='1px solid #ddd', height='260px', overflow='auto', width='50%'))
out_tbl_2 = W.Output(layout=W.Layout(border='1px solid #ddd', height='260px', overflow='auto', width='50%'))

title = W.HTML("<h3 style='margin:6px 0'>AI Marketing Simulation — Live</h3>")
desc  = W.HTML("<p style='margin:0 0 8px 0'>Accuracy lines (one per segment) — final plot renders after the run • Per-segment table • Historical hooks (top per segment)</p>")

# --- Day indicator widgets (NEW) ---
try:
    # If 'w' exists and has simulation_duration, use it; else None
    _total_days_hint = int(getattr(w, "simulation_duration", None))  # noqa: F821
except Exception:
    _total_days_hint = None

day_prog = W.IntProgress(
    value=0,
    min=0,
    max=(int(_total_days_hint) if _total_days_hint else 1),
    layout=W.Layout(width='260px')
)
day_text = W.HTML("<b>Day 0 / ?</b>")
day_row = W.HBox([W.Label("Day:"), day_prog, day_text], layout=W.Layout(align_items='center', gap='8px'))

dashboard = W.VBox(
    [title, desc, day_row, out_plot, W.HBox([out_tbl_1, out_tbl_2], layout=W.Layout(gap='10px'))],
    layout=W.Layout(gap='10px')
)
display(dashboard)

# Placeholder in the plot box
with out_plot:
    clear_output(wait=True)
    print("⏳ The accuracy chart will render here once the simulation completes...")

# ======================
# 2) State (buffering, no live drawing)
# ======================
x_days = []                          # shared x-axis (days)
segment_order = []                   # stable segment order (up to 5)
ys_by_segment = {}                   # seg -> list of y values over days

last_snapshot = None
rolling_snapshots = deque(maxlen=200)

# ======================
# 3) Helpers
# ======================
def _init_segments_from_snapshot(snapshot: dict):
    """Pick up to 5 segment names from any available field and initialize buffers."""
    global segment_order, ys_by_segment
    if segment_order:
        return

    seg_names = []
    seg_acc = snapshot.get("segment_accuracy", {}) or {}
    if seg_acc:
        seg_names = list(seg_acc.keys())
    if not seg_names:
        seg_names = list((snapshot.get("current_hooks") or {}).keys())
    if not seg_names:
        seg_names = list((snapshot.get("segment_visit_metrics") or {}).keys())
    if not seg_names:
        seg_names = list((snapshot.get("data_summary", {}).get("segments_tracked") or []))

    if seg_names:
        picked = list(sorted(seg_names))[:5]  # allow up to 5 segments
        segment_order[:] = picked
        for seg in segment_order:
            ys_by_segment.setdefault(seg, [])

def _top_hooks_per_segment(historical_entries, topk=5):
    """
    historical_entries: list of dicts with keys:
        'segment', 'hook', 'accuracy_score' (optional keys ignored)
    Returns a DataFrame with per-segment top hooks (by best accuracy).
    """
    if not historical_entries:
        return pd.DataFrame(columns=["segment", "hook", "best_accuracy", "count"])

    df = pd.DataFrame(historical_entries)

    # Ensure required columns exist
    for col in ["segment", "hook", "accuracy_score"]:
        if col not in df.columns:
            return pd.DataFrame(columns=["segment", "hook", "best_accuracy", "count"])

    # Coerce numerics
    df["accuracy_score"] = pd.to_numeric(df["accuracy_score"], errors="coerce")

    # Aggregate per (segment, hook)
    agg = (
        df.groupby(["segment", "hook"], dropna=False)
        .agg(best_accuracy=("accuracy_score", "max"),
             count=("hook", "size"))
        .reset_index()
    )

    # Rank within each segment by best_accuracy desc, then count desc
    agg = agg.sort_values(["segment", "best_accuracy", "count"],
                          ascending=[True, False, False])
    agg["rank_in_segment"] = agg.groupby("segment").cumcount() + 1

    # Keep top-k per segment
    top = (
        agg[agg["rank_in_segment"] <= topk]
        .sort_values(["segment", "best_accuracy", "count"], ascending=[True, False, False])
        .reset_index(drop=True)
    )

    # Final columns (no last_seen_day)
    return top[["segment", "hook", "best_accuracy", "count"]]

def refresh_tables():
    # Table 1: per-segment accuracy + current hook (latest)
    with out_tbl_1:
        clear_output(wait=True)
        if not last_snapshot:
            display(pd.DataFrame({"status":["waiting for data"]}))
        else:
            seg_acc   = last_snapshot.get("segment_accuracy", {}) or {}
            cur_hooks = last_snapshot.get("current_hooks", {}) or {}
            segments = segment_order if segment_order else sorted(set(seg_acc) | set(cur_hooks))
            rows = []
            for seg in segments:
                rows.append({
                    "segment": seg,
                    "accuracy": seg_acc.get(seg, None),
                    "current_hook": cur_hooks.get(seg, None),
                })
            df = pd.DataFrame(rows, columns=["segment","accuracy","current_hook"])
            display(df.style.set_caption("Per-segment (latest): accuracy & current hook"))

    # Table 2: TOP hooks per segment from history (using historical_hooks_and_scores)
    with out_tbl_2:
        clear_output(wait=True)
        if not last_snapshot:
            display(pd.DataFrame({"status":["waiting for data"]}))
        else:
            hist = last_snapshot.get("historical_hooks_and_scores", []) or []
            top_df = _top_hooks_per_segment(hist, topk=5)
            if top_df.empty:
                display(pd.DataFrame({"status":["no historical hooks yet"]})
                        .style.set_caption("Historical hooks — top per segment"))
            else:
                show_df = top_df[["segment", "hook", "best_accuracy", "count"]].copy()
                display(show_df.head(25).style.set_caption("Historical hooks — top per segment (by best accuracy)"))

# ======================
# 4) Callback (buffer data only; tables update live)
# ======================
def on_progress(snapshot: dict):
    """Buffer data only; no live plotting. Keeps tables updating."""
    global last_snapshot

    if not isinstance(snapshot, dict):
        return

    # Initialize segment list from any available field
    _init_segments_from_snapshot(snapshot)

    # If still no segments, just update tables and bail
    if not segment_order:
        last_snapshot = snapshot
        refresh_tables()
        return

    # x axis (day)
    day_raw = snapshot.get("day_number", snapshot.get("day", len(x_days)))
    try:
        day = int(day_raw)
    except Exception:
        day = len(x_days)
    x_days.append(day)

    # ---- (NEW) update day UI indicator ----
    # If days are 0-based in data, present 1-based in UI
    disp_day = day if day >= 1 else day + 1
    # If we had a total-days hint, keep it; otherwise the bar will use observed days
    if day_prog.max <= 1 and isinstance(_total_days_hint, int):
        day_prog.max = max(int(_total_days_hint), 1)
    # Never exceed max
    if day_prog.max <= 1:
        # unknown total yet, show open-ended with current observed
        day_prog.max = max(disp_day, 1)
    day_prog.value = min(disp_day, day_prog.max)
    day_text.value = f"<b>Day {disp_day} / {day_prog.max if (_total_days_hint or day_prog.max > 1) else '…'}</b>"

    # y values per segment (buffer only)
    seg_acc = snapshot.get("segment_accuracy", {}) or {}
    for seg in segment_order:
        val = seg_acc.get(seg, float("nan"))
        try:
            y = float(val)  # robust for numpy/pandas numerics too
        except Exception:
            y = float("nan")
        ys_by_segment[seg].append(y)

    # Update tables live
    last_snapshot = snapshot
    rolling_snapshots.append(snapshot)
    refresh_tables()

# Show empty tables initially
refresh_tables()

# ======================
# 5) Final plot rendering (after simulation completes)
# ======================
def render_final_plot():
    """Render the accuracy lines once after the run completes, inside out_plot."""
    with out_plot:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7,3))

        if not x_days or not segment_order:
            ax.set_title("No data to plot")
            ax.set_xlabel("Day"); ax.set_ylabel("Accuracy")
            display(fig)
            return

        # Plot each buffered series (truncate to common length if needed)
        n_min = min([len(x_days)] + [len(ys_by_segment.get(s, [])) for s in segment_order])
        for seg in segment_order:
            ys = ys_by_segment.get(seg, [])
            ax.plot(x_days[:n_min], ys[:n_min], lw=2, label=seg)

        ax.set_xlabel("Day")
        ax.set_ylabel("Accuracy")
        ax.set_title("Accuracy per segment (final)")
        ax.legend(loc='best')
        plt.tight_layout()
        display(fig)

# ======================
# 6) Run the simulation and then draw once
# ======================
print("🚀 Starting AI-Powered Retail Marketing Simulation...")
print(f"📊 Simulating {len(all_customers)} customers over {w.simulation_duration} days")
print(f"🔄 OPRO optimization cycles every {w.opro_cycle_days} days")
try:
    print(f"🤖 Using mode: {'OPRO2 (Direct Hook Optimization)' if use_opro2_mode else 'Original OPRO (Prompt Optimization)'}")
except NameError:
    pass
print("=" * 60)

try:
    # Execute the main simulation (buffers data via on_progress)
    w.runSimulation(on_progress)

    print("\n🎉 Simulation completed successfully!")
    print("✅ All marketing campaigns executed")
    print("✅ OPRO optimization cycles completed")
    print("✅ Customer responses processed")

    # Final metrics (optional)
    try:
        final_metrics = w.get_daily_metrics()
        print(f"📈 Final day metrics: {final_metrics}")
    except Exception as metrics_error:
        print(f"⚠️ Could not retrieve final metrics: {metrics_error}")

    # Snap the Day indicator to "done" (NEW)
    try:
        total_days = int(getattr(w, "simulation_duration", None))
    except Exception:
        total_days = None

    if isinstance(total_days, int):
        day_prog.max = max(total_days, 1)
        day_prog.value = day_prog.max
        day_text.value = f"<b>Day {day_prog.max} / {day_prog.max}</b>"
    else:
        observed = len(x_days) if x_days else 0
        day_prog.max = max(observed, 1)
        day_prog.value = observed
        day_text.value = f"<b>Day {observed} / {observed if observed else '…'}</b>"

    # Render the chart once, inside the plot box
    render_final_plot()

except NotImplementedError as nie:
    print(f"\n❌ Implementation Error: {nie}")
    print("💡 You need to implement the required functions before the simulation can run:")
    print("   • HookEvaluator.cosine_similarity")
    print("   • HookEvaluator.evaluate_hook_quality_with_embedding")
    print("   • MarketingSession.run_session")
    print("   • MarketingSession.run_step")
    print("   • WorldSimulatorP1.calculate_segment_hook_accuracy")
    print("   • Opro.optimize_prompt")
    print("\n🔧 Please complete the implementation in the cells above and re-run this cell.")

except Exception as e:
    print(f"\n❌ Simulation Error: {e}")
    import traceback
    print("📋 Full error details:")
    traceback.print_exc()
    print("\n💭 Common issues:")
    print("   • Missing function implementations (see cells above)")
    print("   • Database connection problems")
    print("   • LLM service configuration issues")
    print("   • Missing data files")

print("\n" + "=" * 60)

🚀 Starting AI-Powered Retail Marketing Simulation...
📊 Simulating 40 customers over 50 days
🔄 OPRO optimization cycles every 5 days
🤖 Using mode: OPRO2 (Direct Hook Optimization)

❌ Implementation Error: Get hook quality score using your custom evaluator
💡 You need to implement the required functions before the simulation can run:
   • HookEvaluator.cosine_similarity
   • HookEvaluator.evaluate_hook_quality_with_embedding
   • MarketingSession.run_session
   • MarketingSession.run_step
   • WorldSimulatorP1.calculate_segment_hook_accuracy
   • Opro.optimize_prompt

🔧 Please complete the implementation in the cells above and re-run this cell.



## 7. Run the Simulation

**Now that all components are initialized, let's run the AI-powered retail simulation!**

The simulation will run for 15 days with 5-day OPRO optimization cycles. You'll see:
- Daily marketing campaigns targeting customer segments
- Customer responses and website visit patterns
- OPRO learning and prompt/hook optimization
- Performance metrics and improvements over time

## 9. Database Cleanup

**Important**: Clean up database connections to avoid resource leaks.

In [16]:
# Clean up database connections - MATCHES main_p1.py
try:
    from models.database import close_database
    close_database()
    print("✅ Database connection closed successfully.")
except Exception as e:
    print(f"⚠️  Warning: Error closing database: {e}")

print("\n🎊 Congratulations! You've completed the AI-Powered Retail Transformation simulation!")
print("🚀 You've experienced how OPRO optimization can improve marketing effectiveness")
print("📈 Next steps: Implement the missing functions to see the full system in action")

✅ Database connection closed successfully.

🎊 Congratulations! You've completed the AI-Powered Retail Transformation simulation!
🚀 You've experienced how OPRO optimization can improve marketing effectiveness
📈 Next steps: Implement the missing functions to see the full system in action
